In [1]:
import sys
sys.path.insert(0,'C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\hp\\20261024T040012-859907fe')
import kernel_identity as _kernel_management
_kernel_management.write_kernel_runtime_identity('C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\hp\\20261024T040012-859907fe',55232)
import local_course_adapter as _local_adapter
_local_adapter.install('C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\hp\\20261024T040012-859907fe\\source\\HaiLong_Colab_Exercises','C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\hp\\20261024T040012-859907fe')


# Google Colab edition

Run the setup cell first, then work from top to bottom. Keep the course ZIP in My Drive. Use a fresh Colab runtime when switching dependency profiles. **Select a GPU runtime before running this notebook.** 

Original fast.ai competition exercise. Upload paddy-disease-classification.zip from Kaggle when prompted. No API credentials are written and nothing is submitted. Batch sizes are reduced for Colab GPU memory; full experiments can take hours.

Source: https://github.com/fastai/course22/blob/230390584ba5b990e20487d68681a9ec5b258eff/09-small-models-road-to-the-top-part-2.ipynb

In [2]:
# Colab setup. Run this cell before the exercise cells.
from pathlib import Path
import os, sys, zipfile

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
QUICK_RUN = False  #@param {type:"boolean"}
ARCHIVE_PATH = "/content/drive/MyDrive/HaiLong_Colab_Exercises_2026-10-03.zip"  #@param {type:"string"}

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive', timeout_ms=300000)
    COURSE_ROOT = Path('/content/HaiLong_Colab_Exercises')
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        archive = Path(ARCHIVE_PATH)
        if not archive.is_file():
            print('Select HaiLong_Colab_Exercises_2026-10-03.zip from your computer.')
            uploaded = files.upload()
            choices = [Path(n) for n in uploaded if n.endswith('.zip')]
            if len(choices) != 1:
                raise ValueError('Upload exactly one course ZIP file.')
            archive = choices[0]
        with zipfile.ZipFile(archive) as z:
            base = Path('/content').resolve()
            for member in z.infolist():
                target = (base / member.filename).resolve()
                if not target.is_relative_to(base) or not member.filename.startswith('HaiLong_Colab_Exercises/'):
                    raise ValueError('This is not the expected course archive.')
            z.extractall(base)
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        raise FileNotFoundError('The ZIP must contain HaiLong_Colab_Exercises/colab_runtime.py.')
else:
    # Also permits local checks without assuming the current notebook directory.
    COURSE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'colab_runtime.py').is_file()), None)
    if COURSE_ROOT is None:
        raise RuntimeError('Open this notebook in Colab or run it inside the extracted course folder.')

os.environ['COURSE_SMOKE'] = '1' if QUICK_RUN else '0'
os.environ['WANDB_MODE'] = 'offline'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(COURSE_ROOT))
from colab_runtime import setup_course, finish_run
# Filled individually for each notebook by the builder.

COLAB_CONTEXT = setup_course(COURSE_ROOT, '03_fastai', 'fastai', '11_paddy_small_models.ipynb', USE_GOOGLE_DRIVE, install=IN_COLAB)

Exercise folder: C:\Users\[LOCAL_USER]\.v\hp\20261024T040012-859907fe\source\HaiLong_Colab_Exercises\03_fastai
Run mode: FULL EXERCISE
Results: C:\Users\[LOCAL_USER]\.v\hp\20261024T040012-859907fe\source\HaiLong_Colab_Exercises\03_fastai\results


## Full exercise evidence

- Run all exercise cells top to bottom in hosted Colab and preserve their outputs, errors and execution counts.
- Keep QUICK_RUN=False; shortened checks or synthetic replacements for real datasets do not count as full original exercises.
- Preserve plots, models, metrics and run configurations in Drive; a final-cell marker alone is insufficient.
- Supply the original competition ZIP after accepting its rules; predictions and submission CSVs are local evidence, not submission receipts.

This cell describes evidence to collect; it does not certify completion. GPU batch caps preserve dataset coverage and epoch counts, but are documented configuration changes.


In [3]:
from colab_runtime import competition_data
iskaggle = False

This is part 2 of the [Road to the Top](https://www.kaggle.com/code/jhoward/first-steps-road-to-the-top-part-1) series, in which I show the process I used to tackle the [Paddy Doctor](https://www.kaggle.com/competitions/paddy-disease-classification) competition, leading to four 1st place submissions. If you haven't already, first check out [part 1](https://www.kaggle.com/code/jhoward/first-steps-road-to-the-top-part-1).

## Going faster

First we'll repeat the steps we used last time to access the data and ensure all the latest libraries are installed:

In [4]:
comp = 'paddy-disease-classification'
path = competition_data(comp)
from fastai.vision.all import *
set_seed(42)

A big issue I noticed last time was that originally I created the notebook on my home PC, and each epoch of the resnet we created took under 20 seconds to run. But on Kaggle they took over 3 minutes each! Whilst Kaggle's GPUs are less powerful than what I've got at home, that doesn't come close to explaining this vast difference in speed.

I noticed when Kaggle was running that the "GPU" indicator in the top right was nearly empty, and the "CPU" one was always full. This strongly suggests that the problem was that Kaggle's notebook was CPU bound by decoding and resizing the images. This is a common problem on machines with poor CPU performance -- and indeed Kaggle only provides 2 virtual CPUs at the time of writing.

We really need to fix this, since we need to be able to iterate much more quickly. What we can do is to simply resize all the images to half their height and width -- which reduces their number of pixels 4x. This should mean an around 4x increase in performance for training small models.

Luckily, fastai has a function which does exactly this, whilst maintaining the folder structure of the data: `resize_images`.

In [5]:
trn_path = Path('sml')

In [6]:
resize_images(path/'train_images', dest=trn_path, max_size=256, recurse=True)

This will give us 192x256px images. Let's take a look:

In [7]:
dls = ImageDataLoaders.from_folder(trn_path, valid_pct=0.2, bs=8, seed=42,
    item_tfms=Resize((256,192)))

dls.show_batch(max_n=3)

In this notebook, we'll be experimenting with a few different architectures and image processing approaches (item and batch transforms). In order to make this easier, we'll put our modeling steps together into a little function which we can pass the architecture, item transforms, and batch transforms to:

In [8]:
def train(arch, item, batch, epochs=5):
    dls = ImageDataLoaders.from_folder(trn_path, seed=42, valid_pct=0.2, bs=8, item_tfms=item, batch_tfms=batch)
    learn = vision_learner(dls, arch, metrics=error_rate).to_fp16()
    learn.fine_tune(epochs, 0.01)
    return learn

Our `item_tfms` already resize our images to small sizes, so this shouldn't impact the accuracy of our models much, if at all. Let's re-run our resnet26d to test.

In [9]:
learn = train('resnet26d', item=Resize(192),
              batch=aug_transforms(size=128, min_scale=0.75))

C:\Users\[LOCAL_USER]\.v\hlpaddy\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\[LOCAL_USER]\.v\hp\20261024T040012-859907fe\hf-cache\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 64.2MB            

model.safetensors: downloading bytes:           |  0.00B            

C:\Users\[LOCAL_USER]\.v\hlpaddy\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\[LOCAL_USER]\.v\hp\20261024T040012-859907fe\hf-cache\hub\models--timm--resnet26d.bt_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Historical split-order hash differs: executing all ten original phases/all37epochs in this new local attempt.
Storage check (before_fit): artifacts 0.000 GB; configured budget 80.000 GB; additional estimate 0.304 GB; runtime free 410.355 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.000 GB; configured budget 80.000 GB; additional estimate 0.073 GB; runtime free 410.355 GB. Budget is not Drive quota.


...

Storage check (checkpoint_write): artifacts 0.396 GB; configured budget 80.000 GB; additional estimate 0.266 GB; runtime free 409.948 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.396 GB; configured budget 80.000 GB; additional estimate 0.266 GB; runtime free 409.949 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.396 GB; configured budget 80.000 GB; additional estimate 0.266 GB; runtime free 409.948 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.396 GB; configured budget 80.000 GB; additional estimate 0.266 GB; runtime free 409.950 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.396 GB; configured budget 80.000 GB; additional estimate 0.073 GB; runtime free 409.950 GB. Budget is not Drive quota.


That's a big improvement in speed, and the accuracy looks fine.

## A ConvNeXt model

I noticed that the GPU usage bar in Kaggle was still nearly empty, so we're still CPU bound. That means we should be able to use a more capable model with little if any speed impact. Let's look again at the options in [The best vision models for fine-tuning](https://www.kaggle.com/code/jhoward/the-best-vision-models-for-fine-tuning). `convnext_small` tops the performance/accuracy tradeoff score there, so let's give it a go!

In [10]:
arch = 'convnext_small.fb_in22k'

In [11]:
learn = train(arch, item=Resize(192, method='squish'),
              batch=aug_transforms(size=128, min_scale=0.75))

model.safetensors: reconstructing file:   0%|          |  0.00B /  265MB            

model.safetensors: downloading bytes:           |  0.00B            

C:\Users\[LOCAL_USER]\.v\hlpaddy\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\[LOCAL_USER]\.v\hp\20261024T040012-859907fe\hf-cache\hub\models--timm--convnext_small.fb_in22k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Storage check (before_fit): artifacts 0.130 GB; configured budget 80.000 GB; additional estimate 0.666 GB; runtime free 409.951 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.130 GB; configured budget 80.000 GB; additional estimate 0.209 GB; runtime free 409.951 GB. Budget is not Drive quota.


epoch,train_loss,valid_loss,error_rate,time
0,1.616032,1.189351,0.342624,01:29


...

Storage check (checkpoint_write): artifacts 1.341 GB; configured budget 80.000 GB; additional estimate 0.812 GB; runtime free 408.730 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 1.341 GB; configured budget 80.000 GB; additional estimate 0.812 GB; runtime free 408.725 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 1.341 GB; configured budget 80.000 GB; additional estimate 0.812 GB; runtime free 408.601 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 1.341 GB; configured budget 80.000 GB; additional estimate 0.812 GB; runtime free 408.584 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 1.341 GB; configured budget 80.000 GB; additional estimate 0.209 GB; runtime free 408.584 GB. Budget is not Drive quota.


Wow our error rate has halved! That's a great result. And, as expected, the speed hasn't gone up much at all. This seems like a great model for iterating on.

## Preprocessing experiments

So, what shall we try first? One thing which can make a difference is whether we "squish" a rectangular image into a square shape by changing it's aspect ratio, or randomly crop out a square from it, or whether we add black padding to the edges to make it a square. In the previous version we "squished". Let's try "crop" instead, which is fastai's default:

In [12]:
learn = train(arch, item=Resize(192),
              batch=aug_transforms(size=128, min_scale=0.75))

Storage check (before_fit): artifacts 0.533 GB; configured budget 80.000 GB; additional estimate 0.666 GB; runtime free 409.392 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.533 GB; configured budget 80.000 GB; additional estimate 0.209 GB; runtime free 409.392 GB. Budget is not Drive quota.


epoch,train_loss,valid_loss,error_rate,time
0,1.768431,1.180293,0.362806,01:20


Storage check (checkpoint_write): artifacts 0.735 GB; configured budget 80.000 GB; additional estimate 0.219 GB; runtime free 409.195 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.946 GB; configured budget 80.000 GB; additional estimate 0.209 GB; runtime free 408.984 GB. Budget is not Drive quota.

Storage check (before_fit): artifacts 0.735 GB; configured budget 80.000 GB; additional estimate 2.446 GB; runtime free 409.195 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.735 GB; configured budget 80.000 GB; additional estimate 0.213 GB; runtime free 409.195 GB. Budget is not Drive quota.


...

Storage check (checkpoint_write): artifacts 1.744 GB; configured budget 80.000 GB; additional estimate 0.812 GB; runtime free 408.006 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 1.744 GB; configured budget 80.000 GB; additional estimate 0.812 GB; runtime free 407.817 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 1.744 GB; configured budget 80.000 GB; additional estimate 0.812 GB; runtime free 407.813 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 1.744 GB; configured budget 80.000 GB; additional estimate 0.812 GB; runtime free 407.813 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 1.745 GB; configured budget 80.000 GB; additional estimate 0.209 GB; runtime free 407.813 GB. Budget is not Drive quota.


That doesn't seem to have made much difference...

We can also try padding, which keeps all the original image without transforming it -- here's what that looks like:

In [13]:
dls = ImageDataLoaders.from_folder(trn_path, valid_pct=0.2, bs=8, seed=42,
    item_tfms=Resize(192, method=ResizeMethod.Pad, pad_mode=PadMode.Zeros))
dls.show_batch(max_n=3)

In [14]:
learn = train(arch, item=Resize((256,192), method=ResizeMethod.Pad, pad_mode=PadMode.Zeros),
      batch=aug_transforms(size=(171,128), min_scale=0.75))

Storage check (before_fit): artifacts 0.937 GB; configured budget 80.000 GB; additional estimate 0.666 GB; runtime free 408.621 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.937 GB; configured budget 80.000 GB; additional estimate 0.209 GB; runtime free 408.621 GB. Budget is not Drive quota.


epoch,train_loss,valid_loss,error_rate,time
0,1.674467,1.204847,0.326285,01:25


Storage check (checkpoint_write): artifacts 1.138 GB; configured budget 80.000 GB; additional estimate 0.219 GB; runtime free 408.415 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 1.349 GB; configured budget 80.000 GB; additional estimate 0.209 GB; runtime free 408.204 GB. Budget is not Drive quota.

Storage check (before_fit): artifacts 1.138 GB; configured budget 80.000 GB; additional estimate 2.446 GB; runtime free 408.415 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 1.138 GB; configured budget 80.000 GB; additional estimate 0.213 GB; runtime free 408.415 GB. Budget is not Drive quota.


...

Storage check (checkpoint_write): artifacts 2.148 GB; configured budget 80.000 GB; additional estimate 0.812 GB; runtime free 407.407 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 2.148 GB; configured budget 80.000 GB; additional estimate 0.812 GB; runtime free 407.693 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 2.148 GB; configured budget 80.000 GB; additional estimate 0.812 GB; runtime free 407.695 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 2.148 GB; configured budget 80.000 GB; additional estimate 0.812 GB; runtime free 407.693 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 2.148 GB; configured budget 80.000 GB; additional estimate 0.209 GB; runtime free 407.693 GB. Budget is not Drive quota.


That's looking like a pretty good improvement.

## Test time augmentation

To make the predictions even better, we can try [test time augmentation](https://nbviewer.org/github/fastai/fastbook/blob/master/07_sizing_and_tta.ipynb#Test-Time-Augmentation) (TTA), which [our book](https://www.amazon.com/Deep-Learning-Coders-fastai-PyTorch/dp/1492045527) defines as:

> *During inference or validation, creating multiple versions of each image, using data augmentation, and then taking the average or maximum of the predictions for each augmented version of the image.*

Before trying that out, we'll first see how to check the predictions and error rate of our model without TTA:

In [15]:
valid = learn.dls.valid
preds,targs = learn.get_preds(dl=valid)

In [16]:
error_rate(preds, targs)

TensorBase(0.0476)

That's the same error rate we saw at the end of training, above, so we know that we're doing that correctly.

Here's what our data augmentation is doing -- if you look carefully, you can see that each image is a bit lighter or darker, sometimes flipped, zoomed, rotated, warped, and/or zoomed:

In [17]:
learn.dls.train.show_batch(max_n=6, unique=True)

If we call `tta()` then we'll get the average of predictions made for multiple different augmented versions of each image, along with the unaugmented original:

In [18]:
tta_preds,_ = learn.tta(dl=valid)

epoch,train_loss,valid_loss,error_rate,time


<div></div>

Let's check the error rate of this:

In [19]:
error_rate(tta_preds, targs)

TensorBase(0.0437)

That's a huge improvement! We'll definitely want to use this for any submission we make!

## Scaling up

Now that we've got a pretty good model and preprocessing approach, let's scale it up to larger images and more epochs. We'll switch back our path to the original un-resized images, and use 12 epochs using our best settings so far, with larger final augmented images:

In [20]:
trn_path = path/'train_images'

In [21]:
learn = train(arch, epochs=12,
              item=Resize((480, 360), method=ResizeMethod.Pad, pad_mode=PadMode.Zeros),
              batch=aug_transforms(size=(256,192), min_scale=0.75))

Storage check (before_fit): artifacts 1.340 GB; configured budget 80.000 GB; additional estimate 0.666 GB; runtime free 408.498 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 1.340 GB; configured budget 80.000 GB; additional estimate 0.209 GB; runtime free 408.498 GB. Budget is not Drive quota.


epoch,train_loss,valid_loss,error_rate,time
0,1.531286,0.953860,0.263335,02:22


Storage check (checkpoint_write): artifacts 1.542 GB; configured budget 80.000 GB; additional estimate 0.219 GB; runtime free 408.297 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 1.753 GB; configured budget 80.000 GB; additional estimate 0.209 GB; runtime free 408.086 GB. Budget is not Drive quota.

Storage check (before_fit): artifacts 1.542 GB; configured budget 80.000 GB; additional estimate 2.446 GB; runtime free 408.297 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 1.542 GB; configured budget 80.000 GB; additional estimate 0.213 GB; runtime free 408.297 GB. Budget is not Drive quota.


...

Storage check (checkpoint_write): artifacts 2.552 GB; configured budget 80.000 GB; additional estimate 0.812 GB; runtime free 407.037 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 2.552 GB; configured budget 80.000 GB; additional estimate 0.812 GB; runtime free 407.032 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 2.552 GB; configured budget 80.000 GB; additional estimate 0.812 GB; runtime free 407.034 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 2.552 GB; configured budget 80.000 GB; additional estimate 0.812 GB; runtime free 407.042 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 2.552 GB; configured budget 80.000 GB; additional estimate 0.209 GB; runtime free 407.042 GB. Budget is not Drive quota.


This is around twice as accurate as our previous best model - let's see how it performs with TTA too:

In [22]:
tta_preds,targs = learn.tta(dl=learn.dls.valid)
error_rate(tta_preds, targs)

epoch,train_loss,valid_loss,error_rate,time


<div></div>

TensorBase(0.0211)

Once again, we get a big boost from TTA. This is one of the most under-appreciated deep learning tricks, in my opinion! (I'm not sure there's any other frameworks that make it quite so easy, so perhaps that's part of the reason why...)

## Submission

We're now ready to get our Kaggle submission sorted. First, we'll grab the test set like we did in the last notebook:

In [23]:
tst_files = get_image_files(path/'test_images').sorted()
tst_dl = learn.dls.test_dl(tst_files)

Next, do TTA on that test set:

In [24]:
preds,_ = learn.tta(dl=tst_dl)

epoch,train_loss,valid_loss,error_rate,time


<div></div>

We need to indices of the largest probability prediction in each row, since that's the index of the predicted disease. `argmax` in PyTorch gives us exactly that:

In [25]:
idxs = preds.argmax(dim=1)

Now we need to look up those indices in the `vocab`. Last time we did that using pandas, although since then I realised there's an even easier way!:

In [26]:
vocab = np.array(learn.dls.vocab)
results = pd.Series(vocab[idxs], name="idxs")

In [27]:
ss = pd.read_csv(path/'sample_submission.csv')
ss['label'] = results
ss.to_csv('results/subm.csv', index=False)
print(ss.head())

     image_id   label
0  200001.jpg   hispa
1  200002.jpg  normal
2  200003.jpg   blast
3  200004.jpg   blast
4  200005.jpg   blast


In [28]:
print("Predictions stay in this notebook. No competition submission or publication is performed.")

Predictions stay in this notebook. No competition submission or publication is performed.


This gets a score of 0.9827, which is well within the top 25% of the competition -- that's a big improvement, and we're still using a single small model!

In [29]:
print("Predictions stay in this notebook. No competition submission or publication is performed.")

Predictions stay in this notebook. No competition submission or publication is performed.


## Conclusion

We've made a big step today, despite just using a single model that trains in under 20 minutes even on Kaggle's rather under-powered machines. Next time, we'll try scaling up to some bigger models and doing some ensembling.

If you found this notebook useful, please remember to click the little up-arrow at the top to upvote it, since I like to know when people have found my work useful, and it helps others find it too. And if you have any questions or comments, please pop them below -- I read every comment I receive!

Save your outputs

Run the following cell after finishing. Choose **File > Save** in Colab to preserve your code and displayed results. Files in `results/` and `models/` persist in Drive when enabled.

In [30]:
finish_run(COLAB_CONTEXT)

Saved run settings. Save the notebook in Colab to keep its cell outputs.


In [31]:
_local_adapter.probe()


__HAILONG_LOCAL_PART1_EVIDENCE__{"backend": "local_native_windows_cuda_with_explicit_completed_phase_lineage", "in_colab": false, "drive_enabled": false, "quick_run": false, "notebook": "03_fastai/11_paddy_small_models.ipynb", "source_sha256": "1142914db00f3ddf4b08ce61737d65b7fc3d7b5eb0d7a531deeae1a6d34a6692", "fits": [{"fit_index": 0, "epochs": 1, "train_rows": 8326, "validation_rows": 2081, "microbatch_size": 8}, {"fit_index": 1, "epochs": 5, "train_rows": 8326, "validation_rows": 2081, "microbatch_size": 8}, {"fit_index": 2, "epochs": 1, "train_rows": 8326, "validation_rows": 2081, "microbatch_size": 8}, {"fit_index": 3, "epochs": 5, "train_rows": 8326, "validation_rows": 2081, "microbatch_size": 8}, {"fit_index": 4, "epochs": 1, "train_rows": 8326, "validation_rows": 2081, "microbatch_size": 8}, {"fit_index": 5, "epochs": 5, "train_rows": 8326, "validation_rows": 2081, "microbatch_size": 8}, {"fit_index": 6, "epochs": 1, "train_rows": 8326, "validation_rows": 2081, "microbatch_size

{'backend': 'local_native_windows_cuda_with_explicit_completed_phase_lineage',
 'in_colab': False,
 'drive_enabled': False,
 'quick_run': False,
 'notebook': '03_fastai/11_paddy_small_models.ipynb',
 'source_sha256': '1142914db00f3ddf4b08ce61737d65b7fc3d7b5eb0d7a531deeae1a6d34a6692',
 'fits': [{'fit_index': 0,
   'epochs': 1,
   'train_rows': 8326,
   'validation_rows': 2081,
   'microbatch_size': 8},
  {'fit_index': 1,
   'epochs': 5,
   'train_rows': 8326,
   'validation_rows': 2081,
   'microbatch_size': 8},
  {'fit_index': 2,
   'epochs': 1,
   'train_rows': 8326,
   'validation_rows': 2081,
   'microbatch_size': 8},
  {'fit_index': 3,
   'epochs': 5,
   'train_rows': 8326,
   'validation_rows': 2081,
   'microbatch_size': 8},
  {'fit_index': 4,
   'epochs': 1,
   'train_rows': 8326,
   'validation_rows': 2081,
   'microbatch_size': 8},
  {'fit_index': 5,
   'epochs': 5,
   'train_rows': 8326,
   'validation_rows': 2081,
   'microbatch_size': 8},
  {'fit_index': 6,
   'epochs': 1,
